# LAB2 Частина 3. Оброблення даних про успішність

Виконавець:  <b>Базилевич Олексій</b>, група <b>К-27</b>
<br> Викладач: <b>Андрій Ляшко</b>

Нам знадобиться бібліотека <code>pandas</code>.
<br>
Бібліотека є сторонньою, отже потребує встановлення. Краще у віртуальне оточення. 
<br>
Мала бути встановлена, якщо ні (клітина з імпортуванням виконується з помилкою), то треба встановити. Можна прямо з Jupyter. Тільки не під час пари. 

<b>Наступну клітину можна розкоментовувати та виконувати тільки, якщо в якості ядра (kernel) вибране віртуальне оточення!</b>

In [137]:
#!pip install pandas

Якщо встановлювати не в оточення, то хоча б тільки для користувача (з ключем --user).

In [138]:
#!pip3.14 install pandas --user    # Linux

In [139]:
#!py -3.14 -m pip install pandas --user    # Windows

In [140]:
from pathlib import Path
import pandas as pd
from scipy.interpolate import insert

print(f'pandas {pd.__version__}')

pandas 3.0.1


### Постановка задачі

У файлі <code>data_new.csv</code> знаходяться дані з поточними результатами студентів за ЛР, що складається з двох частин. Відомо, що файли записаний у форматі csv з кодуванням utf-8, може містити білі рядки та містить заголовок. Також відомо, що інформація про жодного студента не записана на кількох рядках одночасно.

<b>Завдання</b>:
З використанням виключно засобів бібліотеки <code>pandas</code>
<ol>
<li>Завантажити наявну у файлах інформацію в об'єкт <code>DataFrame</code>.</li>
    Заголовками стовпчиків мають бути: <code>group</code>, <code>name</code>, <code>var</code>, <code>part1</code>, <code>part2</code>.<br>
<li>Знайти про скількох студентів наявна інформація у файлі.</li>
<li>Знайти студентів, хто здав обидві частини. Скільки таких студентів?</li>
Цікавить тільки група, ПІБ, номер варіанту, загальна кількість балів. 
Тут і далі: здали = отримали бали, можливо нулеві.
<li>Знайти тих студентів, хто не здав жодної частини. Скільки таких студентів?</li>
     Цікавить тільки група, ПІБ, номер варіанту. 
<li>Знайти тих студентів, хто здав першу частину.</li>
Цікавить тільки група, ПІБ, номер варіанту, бали за першу частину. 
 <li>Знайти тих студентів групи К14, що здали хоч одну частину лабораторної. Цікавить уся наявна інформація.</li>  
<li>Для кожної групи знайти кількість студентів та кількість студентів, що щось здали.</li>
</ol>

<b>Крок 1</b>
Завантажити наявну у файлах інформацію в об'єкт <code>DataFrame</code>.

Заголовками стовпчиків мають бути: <code>group</code>, <code>name</code>, <code>var</code>, <code>part1</code>, <code>part2</code>.

Задаємо шлях до файлу <code>data_new.csv</code>.

In [141]:
path = Path('data_new.csv')

Завантажимо дані за допомогою функції <code>pd.read_csv</code> (<a href="https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html#pandas-read-csv" target=_blank rel="noopener noreferrer">документація тут</a>).

Зверніть увагу на те, що:
<ol>
    <li>файл записано в кодуванні <code>utf-8<code></li>
    <li>перший рядок файлу містить заголовки, проте під час завантаження слід підставити власні заголовки group, name, var, part1, part2</li>
    <li>значення в деяких стовпчиках можуть бути відсутні</li>
</ol>
    
<details>
    <summary>Підказки.</summary>
    Функції варто передати параметри, що задають:
    <ol>
    <li>кодування, в якому записано текстовий файл</li>
    <li>номер рядка, в якому записано заголовки</li>
    <li>тип, в якому слід записати в пам'ять останній стовпчик</li>
    <li>назви стовпчиків</li>
</ol>
</details>

<details>
    <summary>Ще більше підказок</summary>
    <ol>
    <li>параметр <code>sep</code> задає роздільники полів</li>
    <li>параметр <code>encoding</code> задає кодування</li>
    <li>параметр <code>header</code> задає номер рядка, в якому записано заголовки (нумерація починається з 0)</li>
    <li>параметр <code>dtype</code> може бути словником, що задає типи стовпчиків (можливо, не всіх); назві стовпчика відповідає назва типу</li>
    <li>тип <code>Int64</code> містить цілі числа та особливе значення <code>NA</code></li>
    <li>параметр <code>names</code> задає назви стовпчиків</li>
</ol>
</details>

In [142]:
data = pd.read_csv(path, encoding='utf8', sep=",", header=0, dtype={"lines": "Int64"}, names=['group', 'name', 'var', 'part1', 'part2'])
data.head()

,group,name,var,part1,part2
0,K10,Прізвище3 Ім'я3,1,NaN,NaN
1,K10,Прізвище9 Ім'я9,2,NaN,NaN
2,K10,Прізвище16 Ім'я16,3,NaN,NaN
3,K10,Прізвище18 Ім'я18,4,NaN,NaN
4,K10,Прізвище25 Ім'я25,5,NaN,9.0


In [143]:
data.tail()

,group,name,var,part1,part2
126,K14,Прізвище112 Ім'я112,132,NaN,NaN
127,K14,Прізвище117 Ім'я117,133,NaN,NaN
128,K14,Прізвище120 Ім'я120,134,NaN,NaN
129,K14,Прізвище126 Ім'я126,135,NaN,NaN
130,K14,Прізвище127 Ім'я127,136,NaN,NaN


Подивимось інформацію про завантажені дані.
Документація на метод <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html" target=_blank rel="noopener noreferrer"><code>info</code></a>.

In [144]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   group   131 non-null    str    
 1   name    131 non-null    str    
 2   var     131 non-null    int64  
 3   part1   13 non-null     float64
 4   part2   13 non-null     float64
dtypes: float64(2), int64(1), str(2)
memory usage: 5.2 KB


<b>Крок 2</b> Знайти про скількох студентів наявна інформація у файлах.

<details>
    <summary>Підказка.</summary>
    Вибрати стовпчик з прізвищами студентів і знайти кількість наявних у ньому значень. Оскільки ПІБ студентів унікальні, то цього буде достатньо.
</details>

In [145]:
data['name'].nunique()

131

<b>Крок 3</b> Знайти студентів, хто здав обидві частини. Скільки таких студентів?

Цікавить тільки група, ПІБ, номер варіанту, загальна кількість балів.
<br>
Тут і далі: здали = отримали бали, можливо нулеві.

<details>
    <summary>Підказка.</summary>
    Вибрати стовпчик з прізвищами студентів і знайти кількість наявних у ньому значень. Оскільки ПІБ студентів унікальні, то цього буде достатньо.
</details>

<b>3.1</b> Для рядків таблиці визначити, чи правда що в рядку обидва поля <code>part1</code>, <code>part2</code> заповнені.<br>
<i>Підказка.</i> Метод 
<a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html?highlight=notna#pandas.DataFrame.notna"
target=_blank rel="noopener noreferrer"> <code>notna</code></a>
      обчислює ознаку наявності значення поля.
        
<details> 
    <summary>Ще підказки</summary>
    <ul>
    <li>Вибрати з таблиці стовпчик <code>part1</code> та для його елементів з'ясувати чи заповнені вони. </li>
    <li>Виконати аналогічне для стовпчика <code>part2</code>.</li>
    <li>Скомбінувати отримані результати (поля обох стовпчиків у рядку мають бути заповнені одночасно).</li>
 </ul>
</details>

In [146]:
ans = data['part1'].notna() & data['part2'].notna()
ans

0      False
1      False
2      False
3      False
4      False
       ...  
126    False
127    False
128    False
129    False
130    False
Length: 131, dtype: bool

<b>3.2</b> Щоб знайти шукану кількість студентів достатньо знайти кількість рядків, для яких обчислена ознака істинна.

<details> 
    <summary>Підказка</summary>
    Логічне значення <code>True</code> інтерпретується як <code>1</code>. Тому можна просумувати отримані логічні значення.
       Метод 
        <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html?highlight=sum#pandas.Series.sum"
target=_blank rel="noopener noreferrer">   <code>sum</code></a>
        стане у нагоді.
   
</details>

In [147]:
count = ans.sum()
count

np.int64(12)

<b>3.3</b> Щоб знайти шуканих студентів, слід профільтрувати загальну таблицю за обчисленою ознакою.<br>
Копію (використати метод <code>copy</code>) отриманого результату зберегти в змінну <code>task3</code>.

Примітка. Надалі до отриманої вибірки будемо додавати стовпчики, тому важливо, щоб вибірка не використовувала дані спільно з первинною таблицею.

In [148]:
task3 = data[ans].copy()
task3.head()

,group,name,var,part1,part2
32,K11,Прізвище0 Ім'я0,38,6.0,9.0
38,K11,Прізвище51 Ім'я51,44,6.0,9.0
46,K11,Прізвище84 Ім'я84,52,6.0,9.0
58,K12,Прізвище12 Ім'я12,64,6.0,9.0
64,K12,Прізвище40 Ім'я40,70,6.0,9.0


<b>3.4</b> До отриманої таблиці додати суму балів під іменем <code>total</code>.

In [149]:
task3.insert(loc=5, column='total', value=task3['part1']+task3['part2'])
task3.head()

,group,name,var,part1,part2,total
32,K11,Прізвище0 Ім'я0,38,6.0,9.0,15.0
38,K11,Прізвище51 Ім'я51,44,6.0,9.0,15.0
46,K11,Прізвище84 Ім'я84,52,6.0,9.0,15.0
58,K12,Прізвище12 Ім'я12,64,6.0,9.0,15.0
64,K12,Прізвище40 Ім'я40,70,6.0,9.0,15.0


<b>3.5</b> З отриманої таблиці вибрати стовпчики <code>group</code>, <code>name</code>, <code>var</code>, <code>total</code>.

In [150]:
task3[['group', 'name', 'var', 'total']]

,group,name,var,total
32,K11,Прізвище0 Ім'я0,38,15.0
38,K11,Прізвище51 Ім'я51,44,15.0
46,K11,Прізвище84 Ім'я84,52,15.0
58,K12,Прізвище12 Ім'я12,64,15.0
64,K12,Прізвище40 Ім'я40,70,15.0
88,K13,Прізвище4 Ім'я4,94,15.0
89,K13,Прізвище6 Ім'я6,95,15.0
92,K13,Прізвище24 Ім'я24,98,15.0
95,K13,Прізвище33 Ім'я33,101,15.0
103,K13,Прізвище97 Ім'я97,109,15.0


<b>Крок 4</b> Знайти тих студентів, хто не здав жодної частини. Скільки таких студентів?

Цікавить тільки група, ПІБ, номер варіанту. 

<details>
<summary>Підказки</summary>
    <ul>
        <li>
     Метод 
        <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html?highlight=isna#pandas.DataFrame.isna" target=_blank rel="noopener noreferrer">   <code>isna</code></a> 
обчислює ознаку відсутності значення поля.
            </li>
        <li>Далі використати вибірку за умовою, а також вибрати тільки потрібні стовпчики.</li>
    </ul>
</details>
  

In [151]:
not_ans = data['part1'].isna() & data['part2'].isna()
task31 = data[not_ans].copy()
task31[['group', 'name', 'var']]

,group,name,var
0,K10,Прізвище3 Ім'я3,1
1,K10,Прізвище9 Ім'я9,2
2,K10,Прізвище16 Ім'я16,3
3,K10,Прізвище18 Ім'я18,4
5,K10,Прізвище30 Ім'я30,6
...,...,...,...
126,K14,Прізвище112 Ім'я112,132
127,K14,Прізвище117 Ім'я117,133
128,K14,Прізвище120 Ім'я120,134
129,K14,Прізвище126 Ім'я126,135


In [152]:
count1 = not_ans.sum()
count1

np.int64(117)

<b>Крок 5</b> Знайти тих студентів, хто здав першу частину.

Цікавить тільки група, ПІБ, номер варіанту, бали за першу частину. 
<details>
<summary>Підказки</summary>
    <ul>
        <li>
     Метод 
        <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html?highlight=isna#pandas.DataFrame.isna" target=_blank rel="noopener noreferrer">   <code>isna</code></a> 
обчислює ознаку відсутності значення поля.
            </li>
        <li>Далі використати вибірку за умовою, а також вибрати тільки потрібні стовпчики.</li>
    </ul>
</details>
  

In [153]:
not_ans1 = data['part1'].notna()
task311 = data[not_ans1].copy()
task311[['group', 'name', 'var', 'part1']]

,group,name,var,part1
32,K11,Прізвище0 Ім'я0,38,6.0
38,K11,Прізвище51 Ім'я51,44,6.0
46,K11,Прізвище84 Ім'я84,52,6.0
58,K12,Прізвище12 Ім'я12,64,6.0
64,K12,Прізвище40 Ім'я40,70,6.0
88,K13,Прізвище4 Ім'я4,94,6.0
89,K13,Прізвище6 Ім'я6,95,6.0
92,K13,Прізвище24 Ім'я24,98,6.0
95,K13,Прізвище33 Ім'я33,101,6.0
103,K13,Прізвище97 Ім'я97,109,6.0


<b>Крок 6</b> Знайти студентів групи К14, що здали хоч одну частину лабораторної. Цікавить уся наявна інформація.

<u>Підказки</u><br>
<ul>
    <li>Знайти рядки таблиці, що стосуються студентів групи К14.</li>
    <li>Знайти рядки таблиці, в яких хоча б один зі стовпчиків <code>part1</code>, <code>part2</code> заповнений.</li>   
    <li>Скомбінувати попередні умови: відповідна вибірка буде шуканою.</li>          
</ul>

<b>6.1</b> Знайти рядки таблиці, що стосуються студентів групи К14.

<details> 
    <summary>Підказка</summary>
    Вибрати рядки таблиці, де значення поля <code>group</code> дорівнює K14.
</details>

In [154]:
k14only = (data['group'] == 'K14')
taskK14only = data[k14only].copy()
taskK14only[['group', 'name', 'var', 'part1', 'part2']]

,group,name,var,part1,part2
110,K14,Прізвище5 Ім'я5,116,6.0,NaN
111,K14,Прізвище7 Ім'я7,117,NaN,NaN
112,K14,Прізвище10 Ім'я10,118,NaN,NaN
113,K14,Прізвище15 Ім'я15,119,NaN,NaN
114,K14,Прізвище21 Ім'я21,120,NaN,NaN
115,K14,Прізвище34 Ім'я34,121,NaN,NaN
116,K14,Прізвище1 Ім'я1,122,NaN,NaN
117,K14,Прізвище47 Ім'я47,123,NaN,NaN
118,K14,Прізвище56 Ім'я56,124,NaN,NaN
119,K14,Прізвище62 Ім'я62,125,6.0,9.0


<b>6.2</b> Знайти рядки таблиці, в яких хоча б один зі стовпчиків <code>part1</code>, <code>part2</code> заповнений.

<details> 
    <summary>Підказки</summary>
    <ul>
    <li>Вибрати з таблиці стовпчики <code>part1</code>, <code>part2</code>.</li>
    <li>В отриманій вибірці для клітин таблиці з'ясувати чи заповнені вони. 
        Метод 
        <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html?highlight=notna#pandas.DataFrame.notna"
target=_blank rel="noopener noreferrer">   <code>notna</code></a>
        стане у нагоді.
        </li>   
    <li>Далі слід знайти, чи наявні в рядках істинні значення:<br>
        можна явно записати умову з використанням АБО;<br>
        можна використати метод 
        <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.any.html?highlight=any#pandas.DataFrame.any"
target=_blank rel="noopener noreferrer">   <code>any</code></a>. 
        Не забудьте врахувати, що перевіряються поля рядка, а не стовпчика. Тому слід правильно вказати вісь, уздовж якої виконується перевірка.    
</ul>
</details>

In [155]:
ans62 = (data['part1'].notna()) | (data['part2'].notna())
task62 = data[ans62].copy()
task62[['group', 'name', 'var', 'part1', 'part2']]

,group,name,var,part1,part2
4,K10,Прізвище25 Ім'я25,5,NaN,9.0
32,K11,Прізвище0 Ім'я0,38,6.0,9.0
38,K11,Прізвище51 Ім'я51,44,6.0,9.0
46,K11,Прізвище84 Ім'я84,52,6.0,9.0
58,K12,Прізвище12 Ім'я12,64,6.0,9.0
64,K12,Прізвище40 Ім'я40,70,6.0,9.0
88,K13,Прізвище4 Ім'я4,94,6.0,9.0
89,K13,Прізвище6 Ім'я6,95,6.0,9.0
92,K13,Прізвище24 Ім'я24,98,6.0,9.0
95,K13,Прізвище33 Ім'я33,101,6.0,9.0


<b>6.3</b> Скомбінувати попередні умови: відповідна вибірка буде шуканою.

In [156]:
k14 = (data['group'] == 'K14')
k14ans = (data['part1'].notna()) | (data['part2'].notna())
taskK14 = data[k14 & k14ans].copy()
taskK14[['group', 'name', 'var', 'part1', 'part2']]

,group,name,var,part1,part2
110,K14,Прізвище5 Ім'я5,116,6.0,NaN
119,K14,Прізвище62 Ім'я62,125,6.0,9.0


<b>Крок 7</b> Для кожної групи знайти кількість студентів та кількість студентів, що щось здали.

<b>7.1</b> Обчислити для кожного студента ознаку, чи здав він хоча б одну з частин, та додати відповідний стовпчик до таблиці. Назвати доданий стовпчик <code>done</code>.

In [157]:
data.insert(5, 'done', data['part1'].notna() | data['part2'].notna())
data.head()

,group,name,var,part1,part2,done
0,K10,Прізвище3 Ім'я3,1,NaN,NaN,False
1,K10,Прізвище9 Ім'я9,2,NaN,NaN,False
2,K10,Прізвище16 Ім'я16,3,NaN,NaN,False
3,K10,Прізвище18 Ім'я18,4,NaN,NaN,False
4,K10,Прізвище25 Ім'я25,5,NaN,9.0,True


<b>7.2</b> З використанням методу <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html?highlight=groupby#pandas.DataFrame.groupby"
target=_blank rel="noopener noreferrer"><code>groupby</code></a> виконати групування по групах. 

Далі за допомогою методу <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html?highlight=agg#pandas.DataFrame.agg"
target=_blank rel="noopener noreferrer"><code>agg</code></a> для кожної групи для стовпчика <code>done</code> обчислити кількість (<code>count</code>) та суму (<code>sum</code>) значень. Це буде кількість студентів групи та кількість студентів групи, що щось здали.<br>
Результат присвоїти змінній <code>res</code>

In [167]:
res = data.groupby('group')['done'].agg(['count', 'sum']).reset_index()
res

,group,count,sum
0,K10,26,1
1,K11,31,3
2,K12,31,2
3,K13,22,6
4,K14,21,2


Перейменуємо заголовки колонок.

In [168]:
res.columns = ('group', 'number', 'works')
res

,group,number,works
0,K10,26,1
1,K11,31,3
2,K12,31,2
3,K13,22,6
4,K14,21,2


Розрахувати кількість студентів, що нічого не здали. Результат обчислення записати в стовпчик <code>lazy</code>.

In [170]:
res['lazy'] = res['number'] - res['works']
res

,group,number,works,lazy
0,K10,26,1,25
1,K11,31,3,28
2,K12,31,2,29
3,K13,22,6,16
4,K14,21,2,19


## Частину 3 зроблено!